# Librerias

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from gensim.models import KeyedVectors, Word2Vec
import gdown
import ast
from sentence_transformers import SentenceTransformer, util
from prettytable import PrettyTable
import unicodedata
from collections import Counter

c:\Users\grima\Desktop\UNR\TUIA\4° cuatri\PROCESAMIENTO DE LENGUAJE NATURAL\PRACTICA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Apertura del csv

In [2]:
def abrir_archivo_csv(ruta_csv):
    """Abre un archivo CSV y devuelve un DataFrame de pandas."""
    try:
        df = pd.read_csv(ruta_csv)
        return df
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {ruta_csv}")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: El archivo {ruta_csv} está vacío")
        return None
    except pd.errors.ParserError:
        print(f"Error: No se pudo parsear el archivo {ruta_csv}")
        return None

# Limpieza del dataset

In [3]:
__file__ = "tp_Grimaldi_Añaños_Marzol_Sánchez_Martinelli.ipynb"

In [4]:
RUTA_PROYECTO = Path(__file__).resolve().parents[1]
RUTA_CSV_ORIGINAL = RUTA_PROYECTO / "Practica1" / "data" / "libros.csv"
RUTA_CSV_LIMPIO = RUTA_PROYECTO / "Practica 2" / "data" / "libros_limpios.csv"
RUTA_MODELO = RUTA_PROYECTO / "Practica 2" / "modelos"

In [5]:
from src.procesamiento_de_texto import (
    acentos_dieresis_virgulilla,
    minusculas,
    puntuacion,
    stopwords,
)

STOPWORDS_ES = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "como",
    "con", "contra", "cual", "cuando", "de", "del", "desde", "donde",
    "dos", "el", "ella", "ellas", "ellos", "en", "entre", "era", "eran",
    "es", "esa", "esas", "ese", "eso", "esos", "esta", "estas", "este",
    "esto", "estos", "fue", "ha", "han", "hasta", "la", "las", "le", "les",
    "lo", "los", "más", "me", "mi", "mis", "muy", "ni", "no", "nos",
    "o", "otra", "otras", "otro", "otros", "para", "pero", "por", "que",
    "se", "sin", "sobre", "su", "sus", "también", "te", "tiene", "tienen",
    "tu", "tus", "un", "una", "unas", "uno", "unos", "y", "ya",
}

COLUMNAS_DE_TEXTO = [
    "titulo",
    "serie",
    "sinopsis",
]

def limpiar_texto(texto):
    """Aplica todas las funciones de procesamiento de texto en orden."""
    texto = minusculas(texto)
    texto = acentos_dieresis_virgulilla(texto)
    texto = puntuacion(texto)
    texto = stopwords(texto, STOPWORDS_ES)
    return texto

def limpiar_libros():
    """Copia el CSV original, limpia sus columnas textuales y crea otro CSV."""
    if not RUTA_CSV_ORIGINAL.exists():
        raise FileNotFoundError(
            f"No se encontró el CSV original: {RUTA_CSV_ORIGINAL}"
        )

    libros = pd.read_csv(RUTA_CSV_ORIGINAL, keep_default_na=False)
    columnas_faltantes = [
        columna for columna in COLUMNAS_DE_TEXTO if columna not in libros.columns
    ]
    if columnas_faltantes:
        raise ValueError(f"Faltan columnas de texto: {columnas_faltantes}")

    for columna in COLUMNAS_DE_TEXTO:
        libros[columna] = libros[columna].map(limpiar_texto)

    libros["autores"] = libros["autores"].apply(ast.literal_eval)
    libros["generos"] = libros["generos"].apply(ast.literal_eval)
    libros["autores"] = libros["autores"].apply(lambda autores: list(map(limpiar_texto, autores)))
    libros["generos"] = libros["generos"].apply(lambda generos: list(map(limpiar_texto, generos)))

    libros.to_csv(RUTA_CSV_LIMPIO, index=False, encoding="utf-8")
    print(f"CSV limpio creado: {RUTA_CSV_LIMPIO}")
    print(f"Registros procesados: {len(libros)}")

In [6]:
limpiar_libros()

CSV limpio creado: C:\Users\grima\Desktop\UNR\TUIA\4° cuatri\PROCESAMIENTO DE LENGUAJE NATURAL\PRACTICA\NLP\Practica 2\data\libros_limpios.csv
Registros procesados: 200


# Elección de modelo

In [7]:
vector_sizes = [50, 100, 200]
windows = [2, 4, 6]
min_counts = [1, 2, 5]
sg = [0,1]  # 0: CBOW, 1: Skip-gram

In [8]:
def word2vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0
):
    """
    Crea y entrena un modelo Word2Vec.

    Arguments:
        sentences: Lista de listas de palabras.
        vector_size: Dimensionalidad de los vectores.
        window: Tamaño de la ventana de contexto.
        min_count: Mínimo número de ocurrencias de una palabra.
        workers: Número de hilos utilizados.
        sg: Tipo de modelo:
            0 = CBOW
            1 = Skip-gram.

    Returns:
        Word2Vec: Modelo Word2Vec entrenado.
    """

    modelo_word2vec = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        sg=sg,
        seed=42
    )

    return modelo_word2vec

## Tokenizar

In [9]:
def tokenizar(dataset: pd.DataFrame, columna: str):
    """
    Tokeniza el texto en la columna especificada del DataFrame.

    Arguments:
        dataset: pd.DataFrame - DataFrame que contiene la columna.
        columna: str - Nombre de la columna que se desea tokenizar.

    Returns:
        pd.Series - Serie de pandas con los textos tokenizados.
    """

    return dataset[columna].apply(
        lambda x: str(x).split()
    )

## Evaluar

In [10]:
def evaluar_pares(modelo, pares):
    """
    Calcula la similitud promedio entre pares de palabras.

    Solo se utilizan los pares cuyas dos palabras
    están presentes en el vocabulario del modelo.

    Arguments:
        modelo: Modelo Word2Vec entrenado.
        pares: Lista de tuplas (palabra1, palabra2).

    Returns:
        float: Similitud promedio.
    """

    resultados = []

    for palabra1, palabra2 in pares:

        if palabra1 in modelo.wv and palabra2 in modelo.wv:

            similitud = modelo.wv.similarity(
                palabra1,
                palabra2
            )

            resultados.append(similitud)

    if not resultados:
        return None

    return np.mean(resultados)

## Entrenamiento de modelos

In [11]:
def entrenar_modelo_word2vec(
    sentences: pd.DataFrame,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0
):
    """
    Entrena un modelo Word2Vec utilizando la columna especificada.

    Arguments:
        sentences: pd.DataFrame - DataFrame que contiene las oraciones ya tokenizadas.
        vector_size: int - Dimensionalidad de los vectores.
        window: int - Tamaño de la ventana de contexto.
        min_count: int - Mínimo número de ocurrencias.
        workers: int - Número de hilos.
        sg: int - Tipo de modelo:
            0 = CBOW
            1 = Skip-gram.

    Returns:
        Word2Vec - Modelo Word2Vec entrenado.
    """

    modelo_word2vec = word2vec(
        sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        sg=sg
    )

    return modelo_word2vec

In [12]:
def entrenar_modelos():

    libros = pd.read_csv(RUTA_CSV_LIMPIO)

    libros["autores"] = libros["autores"].apply(
        ast.literal_eval
    )

    libros["generos"] = libros["generos"].apply(
        ast.literal_eval
    )

    COLUMNAS_DE_TEXTO = [
        # "titulo",
        "serie",
        "sinopsis",
    ]

    modelos_word2vec = {}

    # Tokenizamos una sola vez.
    # No es necesario hacerlo en cada iteración.
    sentences = tokenizar(
        libros,
        "sinopsis"
    )

    # Probamos diferentes hiperparámetros
    for vector_size in vector_sizes:

        for window in windows:

            for min_count in min_counts:

                for sg_value in sg:

                    print(
                        f"Entrenando modelo con "
                        f"vector_size={vector_size}, "
                        f"window={window}, "
                        f"min_count={min_count}, "
                        f"sg={sg_value}"
                    )

                    modelo_word2vec = entrenar_modelo_word2vec(
                        sentences,
                        vector_size=vector_size,
                        window=window,
                        min_count=min_count,
                        sg=sg_value
                    )

                    nombre_modelo = (
                        f"modelo_word2vec_"
                        f"{vector_size}_"
                        f"{window}_"
                        f"{min_count}_"
                        f"{sg_value}"
                    )

                    modelos_word2vec[
                        nombre_modelo
                    ] = modelo_word2vec

    return modelos_word2vec

## Seleción del modelo

In [13]:
def seleccionar_mejor_modelo(modelos_word2vec, pares):
    """
    Selecciona el mejor modelo Word2Vec basado en la similitud promedio
    de los pares de palabras.

    Arguments:
        modelos_word2vec: Diccionario con los modelos Word2Vec entrenados.
        pares: Lista de tuplas (palabra1, palabra2).

    Returns:
        tuple: (nombre del mejor modelo, similitud promedio)
    """

    promedios = evaluar_modelos(modelos_word2vec, pares)

    if not promedios:
        print("No se encontraron palabras en el vocabulario de los modelos.")
        return None, None

    mejor_modelo = max(promedios, key=promedios.get)
    mejor_similitud = promedios[mejor_modelo]

    return mejor_modelo, mejor_similitud

## Guardar modelo y metricas

In [14]:
def guardar_modelo(modelo, nombre_modelo):
    """
    Guarda el modelo Word2Vec en un archivo.

    Arguments:
        modelo: Modelo Word2Vec entrenado.
        nombre_modelo: Nombre del archivo donde se guardará el modelo.
    """
    ruta_modelo = str(RUTA_MODELO / f"{nombre_modelo}.model")

    modelo.save(ruta_modelo)
    print(f"Modelo guardado en: {ruta_modelo}")

def guardar_combinación_parametros(nombre_modelo):
    """
    Guarda la combinación de parámetros del mejor modelo en un archivo de texto.

    Arguments:
        nombre_modelo: Nombre del archivo donde se guardará la combinación de parámetros.
    """
    ruta_parametros = RUTA_MODELO / "mejor_modelo_parametros.txt"
    with open(ruta_parametros, "w", encoding="utf-8") as f:
        f.write(
            f"Mejor modelo: {nombre_modelo}\n"
            f"Parámetros: {nombre_modelo.split('_')[2:]}\n"
        )
    print(f"Combinación de parámetros guardada en: {ruta_parametros}")

## Carga del modelo

In [15]:
def cargar_modelo(nombre_modelo):
    """
    Carga un modelo Word2Vec desde un archivo.

    Arguments:
        nombre_modelo: Nombre del archivo del modelo a cargar.

    Returns:
        Word2Vec: Modelo Word2Vec cargado.
    """

    ruta_modelo = str(RUTA_MODELO / f"{nombre_modelo}.model")
    if os.path.exists(ruta_modelo):
        modelo = Word2Vec.load(ruta_modelo)
        print(f"Modelo cargado desde: {ruta_modelo}")
        return modelo
    else:
        print(f"No se encontró el modelo en: {ruta_modelo}")
        return None

## Evaluar Modelos

In [16]:
def evaluar_vecinos(modelo, palabras, topn=5):
    """
    Obtiene los vecinos más similares de una lista de palabras.

    Arguments:
        modelo: Modelo Word2Vec o KeyedVectors.
        palabras: Lista de palabras.
        topn: Cantidad de vecinos a devolver.

    Returns:
        dict: Diccionario donde cada palabra tiene asociados sus vecinos
              más similares y su similitud.
    """

    resultados = {}

    for palabra in palabras:

        if palabra not in modelo.key_to_index:
            resultados[palabra] = None
            continue

        vecinos = modelo.most_similar(palabra, topn=topn)

        resultados[palabra] = vecinos

    return resultados

In [17]:
def evaluar_modelos(modelos_word2vec, pares):
    """
    Evalúa todos los modelos Word2Vec entrenados utilizando los pares de palabras.

    Arguments:
        modelos_word2vec: Diccionario con los modelos Word2Vec entrenados.
        pares: Lista de tuplas (palabra1, palabra2).

    Returns:
        dict: Diccionario con la similitud promedio para cada modelo.
    """

    promedios = {}

    for nombre_modelo, modelo in modelos_word2vec.items():
        print(f"Evaluando modelo {nombre_modelo}")
        resultados = evaluar_pares(modelo, pares)
        if resultados is not None:
            promedios[nombre_modelo] = resultados

        else:
            print("No se encontraron palabras en el vocabulario del modelo.")

    return promedios

## Ejecución

In [18]:
def main():
    modelos = entrenar_modelos()

    pares = [
        ("abuelo", "nieto"),
        ("padre", "hijo"),
        ("rey", "reina"),
        ("guerra", "batalla"),
        ("amor", "corazón"),
        ("viaje", "aventura"),
    ]

    mejor_modelo, mejor_similitud = seleccionar_mejor_modelo(modelos, pares)

    if mejor_modelo:
        print(f"El mejor modelo es: {mejor_modelo}, con una similitud promedio de: {mejor_similitud}")
        guardar_modelo(modelos[mejor_modelo], mejor_modelo)
        guardar_combinación_parametros(mejor_modelo)
    else:
        print("No se pudo determinar el mejor modelo.")

In [19]:
main()

Entrenando modelo con vector_size=50, window=2, min_count=1, sg=0
Entrenando modelo con vector_size=50, window=2, min_count=1, sg=1
Entrenando modelo con vector_size=50, window=2, min_count=2, sg=0
Entrenando modelo con vector_size=50, window=2, min_count=2, sg=1
Entrenando modelo con vector_size=50, window=2, min_count=5, sg=0
Entrenando modelo con vector_size=50, window=2, min_count=5, sg=1
Entrenando modelo con vector_size=50, window=4, min_count=1, sg=0
Entrenando modelo con vector_size=50, window=4, min_count=1, sg=1
Entrenando modelo con vector_size=50, window=4, min_count=2, sg=0
Entrenando modelo con vector_size=50, window=4, min_count=2, sg=1
Entrenando modelo con vector_size=50, window=4, min_count=5, sg=0
Entrenando modelo con vector_size=50, window=4, min_count=5, sg=1
Entrenando modelo con vector_size=50, window=6, min_count=1, sg=0
Entrenando modelo con vector_size=50, window=6, min_count=1, sg=1
Entrenando modelo con vector_size=50, window=6, min_count=2, sg=0
Entrenando

# Word2Vec

## Descargar modelo SBW

In [20]:
# ruta_modelo = 'NLP/Practica 2/modelos/SBW-vectors-300-min5.bin.gz'
url = "https://drive.google.com/uc?id=1V8hNcnGEyrz0c_dA-v0_5sg31bNgy75r"

def descarga_modelo(ruta_modelo):
    """
    Descarga el modelo SBW (Spanish Billion Words) desde Google Drive y lo guarda en la ruta especificada.
    Crea los directorios necesarios si no existen.
    """
    os.makedirs(os.path.dirname(ruta_modelo), exist_ok=True)
    gdown.download(url, ruta_modelo, quiet=False)

In [21]:
def SBW_modelo(ruta_modelo):
    """
    Carga el modelo SBW (Spanish Billion Words) desde la ruta especificada.
    Si el modelo no existe, lo descarga primero.
    Returns:
        KeyedVectors: El modelo SBW cargado.
    """
    if not os.path.exists(ruta_modelo):
        descarga_modelo(ruta_modelo)

    model = KeyedVectors.load_word2vec_format(ruta_modelo, binary=True)
    return model

## Parametros

In [22]:
def parametros_word2vec(ruta_parametros):
    with open(ruta_parametros, "r", encoding="utf-8") as archivo:
        lineas = archivo.readlines()
        for linea in lineas:
            print(linea.strip())

    modelo, parametros = None, None

    for linea in lineas:
        if "Mejor modelo" in linea:
            modelo = linea.split(":")[1].strip()

        if "Parámetros" in linea:
            parametros = linea.split(":")[1].strip()

    parametros = ast.literal_eval(parametros)
    return modelo, parametros

## Palabras del dominio

In [23]:
def palabras_dominio(libros):
    palabras = []

    for texto in libros["sinopsis"]:
        palabras.extend(str(texto).split())

    frecuencias = Counter(palabras)

    return [palabra for palabra, frecuencia in frecuencias.most_common(4)]

## Bucle principal para esta sección

In [24]:
def bucle_principal(ruta_csv, ruta_parametros, ruta_modelos):
    libros = abrir_archivo_csv(ruta_csv)
    libros["autores"] = libros["autores"].apply(ast.literal_eval)
    libros["generos"] = libros["generos"].apply(ast.literal_eval)
    COLUMNAS_DE_TEXTO = [
        # "titulo", # No se si tokenizar el titulo, ya que es una columna que se puede buscar por titulo
        "serie",
        "sinopsis",
    ]

    # for columna in COLUMNAS_DE_TEXTO:
    #     libros[columna] = tokenizar(libros, columna)

    sentences = tokenizar(libros, "sinopsis")

    modelo, parametros = parametros_word2vec(ruta_parametros)
    ruta_modelo_guardado = os.path.join(ruta_modelos, modelo + ".model")
    if os.path.exists(ruta_modelo_guardado):
        modelo_word2vec = Word2Vec.load(ruta_modelo_guardado)

    ruta_sbw = os.path.join(ruta_modelos, "SBW-vectors-300-min5.bin.gz")

    modelo_sbw = SBW_modelo(ruta_sbw)

    palabras_frecuentes = palabras_dominio(libros)

    resultados_word2vec = evaluar_vecinos(modelo_word2vec.wv, palabras_frecuentes, topn=5)
    resultados_sbw = evaluar_vecinos(modelo_sbw,palabras_frecuentes, topn=5)

    print("Resultados Word2Vec:")
    for palabra, vecinos in resultados_word2vec.items():
        print(f"  {palabra}: {vecinos}")

    print("Resultados SBW:")
    for palabra, vecinos in resultados_sbw.items():
        print(f"  {palabra}: {vecinos}")

In [25]:
bucle_principal(RUTA_CSV_LIMPIO, RUTA_MODELO / "mejor_modelo_parametros.txt", RUTA_MODELO)

Mejor modelo: modelo_word2vec_200_6_5_1


Parámetros: ['200', '6', '5', '1']
Resultados Word2Vec:
  mas: [('hacia', 0.9991937875747681), ('mujer', 0.9991579651832581), ('encontrar', 0.9991490840911865), ('companeros', 0.9991371631622314), ('personaje', 0.999136209487915)]
  novela: [('tan', 0.9990586042404175), ('isla', 0.9990181922912598), ('reune', 0.9990165829658508), ('cuentos', 0.9989945888519287), ('camino', 0.9989843368530273)]
  historia: [('personaje', 0.99912428855896), ('parece', 0.9990322589874268), ('mil', 0.9990310072898865), ('durante', 0.9990183115005493), ('pasado', 0.9990133047103882)]
  aventuras: [('lugar', 0.9991583824157715), ('libro', 0.999109148979187), ('hombres', 0.9991040229797363), ('ejercito', 0.999096155166626), ('misterio', 0.9990910291671753)]
Resultados SBW:
  mas: [('más', 0.712834894657135), ('ademas', 0.6403582692146301), ('tambien', 0.6285914778709412), ('dandose', 0.603079617023468), ('asi', 0.6004549264907837)]
  novela: [('novelas', 0.7859946489334106), ('autobiográfica', 0.756596446037

El modelo Word2Vec entrenado sobre el corpus de libros presenta similitudes coseno considerablemente superiores a las obtenidas por el modelo SBW. Sin embargo, estos valores no son directamente comparables entre ambos modelos, debido a que fueron entrenados sobre corpus y espacios vectoriales diferentes.

En cuanto a las palabras recuperadas, Word2Vec parece capturar principalmente relaciones asociadas al contexto específico del corpus literario utilizado. Por ejemplo, para "aventuras" obtiene términos como "lugar", "hombres", "ejército" y "misterio", mientras que para "historia" aparecen "personaje" y "pasado". Esto indica que el modelo está aprendiendo asociaciones contextuales características de las obras analizadas.

Por otro lado, SBW presenta relaciones semánticas más generales del idioma español. Para "novela" aparecen términos como "cuento", "narrativa" y "relato", mientras que para "aventuras" aparecen "peripecias", "andanzas" y "desventuras", que presentan una relación semántica más directa con la palabra consultada.

Por lo tanto, el modelo entrenado con el corpus propio parece estar más adaptado al dominio específico de los libros utilizados, mientras que SBW presenta representaciones más generales del lenguaje. La comparación muestra así la diferencia entre un modelo entrenado sobre un corpus específico y un modelo preentrenado sobre un corpus mucho más amplio.